# PDF RAG with Re-ranking

개선 사항:
- **프롬프트 개선**: 문서 특화 / 출처(페이지) 명시 / 역할 명확화
- **Re-ranking 추가**: CrossEncoder로 검색된 청크를 재순위화하여 정확도 향상
- **Retriever 개선**: 초기 k를 크게 잡고 re-ranking 후 상위 N개만 LLM에 전달


In [1]:
# 필요 패키지 설치
# sentence-transformers: CrossEncoder re-ranking에 사용
! uv pip install langchain-community pypdf sentence-transformers

Using Python 3.12.12 environment at: C:\Users\jino7\Desktop\llm_engineering-main\.venv
Audited 3 packages in 19ms


In [2]:
from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import CrossEncoder
import gradio as gr
import os

In [3]:
from dotenv import load_dotenv
load_dotenv(override=True)

True

## 1. 데이터 전처리

In [4]:
PDF_PATH = "test.pdf"

loader = PyPDFLoader(PDF_PATH)
documents = loader.load()

for doc in documents:
    doc.metadata["source_file"] = os.path.basename(PDF_PATH)

print(f"총 {len(documents)}페이지를 불러왔습니다")

총 11페이지를 불러왔습니다


In [5]:
# chunk_size를 줄여 더 세밀한 청크 생성 → re-ranking 효과 극대화
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)

print(f"총 {len(chunks)}개의 청크로 분할되었습니다")
print(f"첫 번째 청크:\n\n{chunks[0]}")

총 17개의 청크로 분할되었습니다
첫 번째 청크:

page_content='2022 
경기 규칙' metadata={'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2022-09-02T03:22:38+09:00', 'author': 'user', 'moddate': '2022-09-02T03:22:38+09:00', 'source': 'test.pdf', 'total_pages': 11, 'page': 0, 'page_label': '1', 'source_file': 'test.pdf'}


## 2. VectorStore 구성

In [6]:
DB_NAME = "pdf_vector_db_large"
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

# 기존 DB 삭제 후 재생성
if os.path.exists(DB_NAME):
    Chroma(persist_directory=DB_NAME, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=DB_NAME
)
print(f"Vectorstore created with {vectorstore._collection.count()} documents")

Vectorstore created with 17 documents


## 3. Re-ranking 설정

**왜 Re-ranking인가?**
- 벡터 검색은 의미적 유사도 기반 → 표면적으로 비슷한 문서가 상위에 올 수 있음
- CrossEncoder는 질문-문서 쌍을 함께 보고 관련도 점수를 계산 → 더 정확한 순위
- 전략: 벡터 검색으로 후보 **k=8** 개를 넓게 뽑고, re-ranking 후 상위 **top_n=3** 개만 LLM에 전달

In [7]:
# 다국어 지원 CrossEncoder 모델 (한국어 포함)
# ms-marco 모델: 질문-문서 관련성 점수 특화 학습
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# 초기 후보 수 (넓게) / re-ranking 후 LLM에 전달할 수 (좁게)
RETRIEVE_K = 8
RERANK_TOP_N = 3

# 초기 retriever: 넓게 검색
retriever = vectorstore.as_retriever(search_kwargs={"k": RETRIEVE_K})


def rerank(question: str, docs: list, top_n: int = RERANK_TOP_N) -> list:
    """CrossEncoder로 문서를 재순위화하고 상위 top_n개 반환"""
    if not docs:
        return []
    
    # (질문, 문서내용) 쌍 생성
    pairs = [(question, doc.page_content) for doc in docs]
    
    # 관련도 점수 계산
    scores = cross_encoder.predict(pairs)
    
    # 점수 기준 내림차순 정렬 후 상위 top_n 반환
    scored_docs = sorted(zip(scores, docs), key=lambda x: x[0], reverse=True)
    
    print("[Re-ranking 결과]")
    for score, doc in scored_docs[:top_n]:
        page = doc.metadata.get('page', '?')
        print(f"  점수: {score:.4f} | 페이지: {page} | 내용 앞부분: {doc.page_content[:60]}...")
    
    return [doc for _, doc in scored_docs[:top_n]]

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

c:\Users\jino7\Desktop\llm_engineering-main\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\jino7\.cache\huggingface\hub\models--cross-encoder--ms-marco-MiniLM-L-6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. F

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

## 4. 개선된 시스템 프롬프트

**기존 프롬프트 문제점:**
- "평가 → 답변 → 메타데이터" 순서 지시 → 사용자 친화적이지 않은 출력
- 지시가 중복/모호

**개선 방향:**
- 문서 기반 답변에 집중하도록 역할 명확화
- 출처(페이지 번호) 자연스럽게 포함
- 문서에 없는 내용은 명확히 불가 표시
- 간결하고 구조적인 답변 유도

In [8]:
SYSTEM_PROMPT_TEMPLATE = """당신은 주어진 문서를 기반으로 질문에 정확하게 답변하는 전문 어시스턴트입니다.

## 답변 규칙
1. **반드시 아래 [참고 문서]의 내용만을 근거로 답변**하세요.
2. 문서에 없는 내용은 "제공된 문서에서 해당 정보를 찾을 수 없습니다."라고 명확히 답하세요.
3. 답변 말미에 근거가 된 **(페이지 번호)**를 괄호 안에 명시하세요. 예: (페이지 3)
4. 답변은 한국어로, 명확하고 간결하게 작성하세요.
5. 여러 규칙이나 항목이 있으면 번호 목록으로 정리하세요.

[참고 문서]
{context}
"""

# 각 청크에 페이지 번호를 포함시켜 컨텍스트 구성
def build_context(docs: list) -> str:
    parts = []
    for doc in docs:
        page = doc.metadata.get('page', '알 수 없음')
        # page는 0-indexed이므로 +1
        parts.append(f"[페이지 {int(page) + 1}]\n{doc.page_content}")
    return "\n\n---\n\n".join(parts)

## 5. LLM 설정

In [9]:
MODEL = "gpt-4.1-nano"
llm = ChatOpenAI(temperature=0, model_name=MODEL)

## 6. RAG 파이프라인 (Re-ranking 포함)

In [10]:
def answer_question(question: str, history) -> str:
    """
    RAG 파이프라인:
    1. 벡터 검색으로 후보 문서 RETRIEVE_K개 검색
    2. CrossEncoder re-ranking으로 상위 RERANK_TOP_N개 선별
    3. 선별된 문서를 컨텍스트로 LLM에 전달
    """
    # Step 1: 벡터 검색 (넓게)
    candidate_docs = retriever.invoke(question)
    
    # Step 2: Re-ranking (좁게)
    reranked_docs = rerank(question, candidate_docs)
    
    # Step 3: 컨텍스트 구성 (페이지 번호 포함)
    context = build_context(reranked_docs)
    
    # Step 4: LLM 호출
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=question)
    ])
    
    return response.content

## 7. 테스트

In [11]:
# 기본 테스트
result = answer_question("경기 규칙을 설명해줘", [])
print(result)

[Re-ranking 결과]
  점수: 8.3588 | 페이지: 4 | 내용 앞부분: 착용해야 합니다. 
 
        경기 시간 
1. U-14 이상의 경기 시간은 전반 후반 각각 30 분...
  점수: 8.2889 | 페이지: 2 | 내용 앞부분: LGFA PLAYING Rules 2022 1 
  
 
 
 
 
 
 
 
소개 
본 소책자는 여자 게일...
  점수: 8.2346 | 페이지: 3 | 내용 앞부분: LGFA PLAYING Rules 2022 2 
 
 
경기장 
1. 여자 게일식 축구는 U-14 경기부터 ...
경기 규칙은 다음과 같습니다:

1. 경기 시간은 U-14 이상 경기의 경우 전반과 후반 각각 30분씩, 총 1시간입니다. (페이지 5)
2. 정규 시간 동안 하프 타임에는 15분을 초과하지 않는 휴식이 허용되며, 연장 시간에는 5분의 하프 타임이 허용됩니다. (페이지 5)
3. 7인제 경기 및 블리츠 경기는 조직위원회가 경기 시간을 결정합니다. (페이지 5)
4. 여자 게일식 축구는 U-14 이상은 GAA 풀사이즈 경기장에서 진행되며, U-13 이하 등급은 경기장 크기가 축소될 수 있습니다. (페이지 4)
5. 15인제 미만 경기의 경우, 경기장 크기, 득점 공간, 경기 시간은 조직 위원회가 단축할 수 있습니다. (페이지 4)
6. 선수는 임신 가능성 또는 뇌진탕 등 건강 문제로 인해 출전할 수 없으며, 출전 시 위험을 감수해야 하고 협회는 책임지지 않습니다. (페이지 4)
7. 복장은 저지, 반바지, 양말, 부츠로 구성되며, 쥬얼리, 귀걸이, 머리핀 등 부상을 유발할 수 있는 아이템 착용이 금지됩니다. (페이지 4)

이상입니다. (페이지 4, 5)


In [12]:
# Re-ranking 효과 확인: 벡터 검색 결과 vs re-ranking 결과 비교
test_question = "선수 교체 규칙은?"

print("=" * 50)
print(f"질문: {test_question}")
print("=" * 50)

# 벡터 검색 결과
candidates = retriever.invoke(test_question)
print(f"\n[벡터 검색 결과 Top-3 (총 {len(candidates)}개 중)]")
for i, doc in enumerate(candidates[:3]):
    page = doc.metadata.get('page', '?')
    print(f"  {i+1}. 페이지 {int(page)+1}: {doc.page_content[:80]}...")

# Re-ranking 결과
print(f"\n[Re-ranking 후 Top-{RERANK_TOP_N}]")
reranked = rerank(test_question, candidates)

print("\n[최종 답변]")
context = build_context(reranked)
system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
response = llm.invoke([
    SystemMessage(content=system_prompt),
    HumanMessage(content=test_question)
])
print(response.content)

질문: 선수 교체 규칙은?

[벡터 검색 결과 Top-3 (총 8개 중)]
  1. 페이지 7: LGFA PLAYING Rules 2022 5 
  
 
교체 & 연장 시간 
1. 인터카운티 경기의 경우, 내셔널 리그에서는 8 강까지 무제한...
  2. 페이지 9: LGFA PLAYING Rules 2022 7 
  
 
 
3.  출혈/두부 손상 의심 교체 선수는 교체 선수로 직접 
교체되었기 때문에, 부...
  3. 페이지 7: 3. 경기 중 주심에 의해 퇴장당한 선수는 교체될 수 없습니다. 
그러나 정규 시간에 퇴장 명령을 받은 선수는 연장 시간에는 
교체될 수 있습니...

[Re-ranking 후 Top-3]
[Re-ranking 결과]
  점수: 8.3338 | 페이지: 6 | 내용 앞부분: 3. 경기 중 주심에 의해 퇴장당한 선수는 교체될 수 없습니다. 
그러나 정규 시간에 퇴장 명령을 받은 선수...
  점수: 8.3140 | 페이지: 7 | 내용 앞부분: 지시에 따라 일시적으로 경기장을 벗어난 다음 추가 
판단에 따라 복귀 여부를 결정해야 합니다. 
 
2. 출...
  점수: 8.2706 | 페이지: 5 | 내용 앞부분: 5. 각 선수의 등번호는 팀 명단에 있는 선수명과 
상응해야 합니다. 
 
6. 팀 명단이 15 명을 초과하...

[최종 답변]
1. 경기 중 주심에 의해 퇴장당한 선수는 교체될 수 없습니다. 다만, 정규 시간에 퇴장 명령을 받은 선수는 연장 시간에는 교체할 수 있습니다. (페이지 7)

2. 연장전이 진행되는 경우, 정규 시간 중 첫 번째 옐로카드로 페널티 박스에 보내진 선수는 페널티 박스에서 경기 시간 10분을 있다가 연장 시간에 다른 선수로 교체될 수 있습니다. (페이지 7)

3. 출혈 부상 또는 두부 손상 의심 선수는 교체될 수 있으며, 이 경우 교체 선수는 관련 사실을 주심에게 서면으로 알려야 합니다. (페이지 8)


## 8. Gradio UI

In [13]:
gr.ChatInterface(
    fn=answer_question,
    title="📄 PDF 문서 QA (Re-ranking 적용)",
    description="문서 기반 질의응답 | 벡터 검색 → CrossEncoder Re-ranking → GPT 답변",
    type="messages"
).launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


[Re-ranking 결과]
  점수: 8.9400 | 페이지: 4 | 내용 앞부분: 착용해야 합니다. 
 
        경기 시간 
1. U-14 이상의 경기 시간은 전반 후반 각각 30 분...
  점수: 8.6718 | 페이지: 2 | 내용 앞부분: LGFA PLAYING Rules 2022 1 
  
 
 
 
 
 
 
 
소개 
본 소책자는 여자 게일...
  점수: 8.6138 | 페이지: 3 | 내용 앞부분: LGFA PLAYING Rules 2022 2 
 
 
경기장 
1. 여자 게일식 축구는 U-14 경기부터 ...
